In [1]:
import PyPDF2
import re
import csv


In [85]:
def extract_archery_results(pdf_path):
    # Lire le PDF
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"

    # Initialiser les données
    archers = []
    
    cat_list = [
    "CL U18F", "CL U21F", "CL S1F", "CL S2F", "CL S3F",
    "CL U18H", "CL U21H", "CL S1H", "CL S2H", "CL S3H",
    "CO U18F", "CO U21F", "CO S1F", "CO S2F", "CO S3F",
    "CO U18H", "CO U21H", "CO S1H", "CO S2H", "CO S3H"
    ]
    # Créer une expression régulière qui correspond à l'une des catégories
    cat_pattern = re.compile(r'|'.join(re.escape(cat) for cat in cat_list))

    # Séparer le texte en lignes
    lines = text.split('\n')

    # Variables pour stocker les informations de l'archer actuel
    current_archer = None

    in_scores = False

    cpt=-1
    for line in lines:
        cpt+=1
        #print(cpt, line)
        line = line.strip()

        # Détecter le nom de l'archer
        name_det_str = "Archer: "
        if name_det_str in line: 
            current_archer = {
                "Nom": line[line.find(name_det_str)+len(name_det_str):], #.strip(),
                "Id_Club": None,
                "Club": None,
                "Catégorie": None,
                "Scores": []
            }

            if current_archer:
                archers.append(current_archer)
            continue

        # Détecter le club et la catégorie
     
        if line.startswith("Clubs / Pays:"):
            if current_archer:
                parts = line.replace("Clubs / Pays:", "").strip().split()
                # Extraire le club (tout ce qui précède la catégorie)
                club_parts = []
                for part in parts:
                    if part.isdigit() or part == "-":
                        club_parts.append(part)
                    else:
                        break
                id_club = " ".join(club_parts).strip()
                id_club = ''.join(re.sub('[-]', '', id_club).split())
                current_archer["Id_Club"] = id_club
                # La catégorie est le reste de la ligne
                nom_club = " ".join(parts[len(club_parts):]).strip()
                cible = re.findall(r'[1-9][ABCD]|[1-9][0-9][ABCD]', nom_club)
                nom_club = re.sub(cible[0], '', nom_club)
                current_archer["Club"] = nom_club
            continue

        # Vérifier si la ligne contient une catégorie
        match = cat_pattern.search(line)
        if match:
            current_archer["Catégorie"] = match.group()
            continue


        # Détecter la catégorie (CO S1H, etc.)
        #if line.startswith("CO"): 
        #    current_archer["Catégorie"] = line#.strip()
        #    continue

        # Détecter la catégorie (CL S1H, etc.)
        #if line.startswith("CL"):
        #    current_archer["Catégorie"] = line#.strip()
        #    continue

        # Détecter les scores après "Somme Tot. 10+XX"
        if "Somme Tot. 10+XX" in line:
            in_scores = True
            cpt_vol = 1
            continue

        # Détecter les scores après "Somme Tot. 10+XX"
        if "Somme Tot. Total10+XX" in line:
            in_scores = True
            cpt_vol = 1
            continue

        # Extraire les lignes de scores
        if in_scores:
            # cas particulier en cas de DNS (pas de score)
            if len(line) == 1:
                current_archer["Scores"].append({
                        "Volée": line,
                        "Score1": "",
                        "Score2": "",
                        "Score3": "",
                        "Score4": "",
                        "Score5": "",
                        "Score6": "",
                    })
                cpt_vol += 2 #on skip 2 lignes car pas de score
            else:
                if cpt_vol % 2 == 1: #volée impaire
                    impair_line = ''.join(line.split())
                    id_volee    = impair_line[0]
                    score_imp   = re.findall(r'10|[1-9XM]', impair_line[1:-1])[:3]
                if cpt_vol % 2 == 0: #volée paire
                    pair_line   = ''.join(line.split())
                    score_pai   = re.findall(r'10|[1-9XM]', pair_line[0:-1])[:3] 
                    # Ajouter les scores pour cette volée
                    current_archer["Scores"].append({
                        "Volée": id_volee,
                        "Score1": score_imp[0],
                        "Score2": score_imp[1],
                        "Score3": score_imp[2],
                        "Score4": score_pai[0],
                        "Score5": score_pai[1],
                        "Score6": score_pai[2]
                    })
                cpt_vol += 1
            if cpt_vol >= 13: #La série est finie
                in_scores = False     
            continue


    return archers

In [86]:
def save_csv_archery_results(results,path_save='./',file_name='results.csv'):

    # Préparer les noms de colonnes pour le CSV
    header = ["Nom", "Id_Club", "Club", "Catégorie"]
    for volee in range(1, 7):
        for score_num in range(1, 7):
            header.append(f"Volee_{volee}_Score_{score_num}")

    # Écrire dans le fichier CSV
    with open(path_save+file_name+".csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)  # Écrire l'en-tête

        # Afficher les résultats
        for archer in results:
        # Préparer les données pour le CSV
            row = [
                archer["Nom"],
                archer["Id_Club"],
                archer["Club"],
                archer["Catégorie"]
            ]
            for volee_data in archer["Scores"]:
                for score_key in ["Score1", "Score2", "Score3", "Score4", "Score5", "Score6"]:
                    row.append(volee_data[score_key])

            writer.writerow(row)    # Écrire les données de l'archer

In [87]:
# Exemple d'utilisation
path_pdf  = "./data_ianseo/" 
path_save = "./data_ianseo/"

file_name = "TAE_2026_D1_etape2_qualif_FCL_HCO"
results = extract_archery_results(path_pdf+file_name+".pdf")
save_csv_archery_results(results, path_save=path_save, file_name=file_name)




In [88]:
file_name = "TAE_2026_D1_etape2_qualif_HCL_FCO"
results = extract_archery_results(path_pdf+file_name+".pdf")
save_csv_archery_results(results, path_save=path_save, file_name=file_name)

In [89]:
file_name = "TAE_2026_D1_etape1_qualif_FCL_HCO"
results = extract_archery_results(path_pdf+file_name+".pdf")
save_csv_archery_results(results, path_save=path_save, file_name=file_name)

In [90]:
file_name = "TAE_2026_D1_etape1_qualif_HCL_FCO"
results = extract_archery_results(path_pdf+file_name+".pdf")
save_csv_archery_results(results, path_save=path_save, file_name=file_name)

In [55]:
1%2

1